In [1]:
from pathlib import Path
import os
import re
import sys
import warnings

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "Data").exists() and (candidate / "Note").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("`Data`와 `Note` 디렉토리를 포함한 프로젝트 루트를 찾지 못했습니다.")

mpl_dir = PROJECT_ROOT / ".mplconfig"
mpl_dir.mkdir(exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(mpl_dir)
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.options.display.float_format = "{:,.4f}".format
sns.set_theme(style="whitegrid")
plt.rcParams["axes.unicode_minus"] = False
try:
    plt.rcParams["font.family"] = "AppleGothic"
except Exception:
    pass

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print(f"Optuna version: {optuna.__version__}")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: /Users/restitutor/Documents/Restitutor/Workspace/Flutter-Python/Python/EP_cycle_stations
Python executable: /usr/local/bin/python3
Optuna version: 4.7.0


In [2]:
df = pd.read_parquet("/Users/restitutor/Documents/Restitutor/Workspace/Flutter-Python/Python/EP_cycle_stations/Data/sort_data/2024_data.parquet")

In [3]:
df.head()

,기준_날짜,시간대,집계_기준,시작_대여소_ID,종료_대여소_ID,전체_건수,전체_이용_분,전체_이용_거리,온도,습도,불쾌지수,강수량,적설량
0,2024-01-01,17,출발시간,ST-479,ST-462,2.0000,0 days 02:31:00,"21,873.0000",4.3000,78,41.9495,0.0000,0.0000
1,2024-01-01,16,출발시간,ST-460,ST-2425,1.0000,0 days 00:11:00,550.0000,6.1000,65,45.8713,0.0000,0.0000
2,2024-01-01,16,출발시간,ST-461,ST-2425,1.0000,0 days 00:15:00,"1,582.0000",6.1000,65,45.8713,0.0000,0.0000
3,2024-01-01,16,출발시간,ST-479,ST-479,1.0000,0 days 00:22:00,"2,526.0000",6.1000,65,45.8713,0.0000,0.0000
4,2024-01-01,17,출발시간,ST-2264,ST-479,1.0000,0 days 00:02:00,445.0000,4.3000,78,41.9495,0.0000,0.0000
